# agent_10 — Financial Sentiment Analysis Agent

## Overview

This notebook implements an **agentic AI workflow** for financial tweet sentiment
classification. Rather than sending a tweet directly to a model and returning a label,
the agent reasons about which model to use, executes that model as a **tool call**,
and explains its decision in plain language — all within a persistent conversational
session that remembers prior turns.

The distinction from a simple LLM prompt is important: the agent has access to
**two real classifiers** trained on this project's data, selects between them based
on the user's intent, and can compare their outputs when a tweet is ambiguous or
the classifiers disagree. This orchestration — choosing between models, routing on
confidence, reconciling disagreements — is what qualifies as an agentic workflow.

## Architecture

```
User message
     │
     ▼
┌─────────────────────────────────────┐
│         DeepSeek LLM Agent          │
│  (instruction-following reasoner)   │
│                                     │
│  Decides which tool to call based   │
│  on user intent and system prompt   │
└────────────┬──────────┬─────────────┘
             │          │
    ┌────────▼──┐  ┌────▼───────────┐
    │classify_  │  │ classify_best  │
    │  fast     │  │(FinBERT fine-  │
    │(TF-IDF +  │  │  tuned)        │
    │  LogReg)  │  │                │
    └────────┬──┘  └────┬───────────┘
             │          │
             └────┬─────┘
                  ▼
         compare_models (optional)
         get_model_info (optional)
                  │
                  ▼
         Agent formulates response
         with explanation + context
```

## Decision Protocol

The agent's system prompt encodes a routing policy that it follows when deciding
which tool to invoke:

| User intent | Tool called |
|---|---|
| Quick or exploratory request | `classify_fast` (TF-IDF) |
| Explicit request for best accuracy | `classify_best` (FinBERT) |
| Ambiguous tweet or verification needed | `compare_models` |
| Fast and best disagree | `compare_models` (auto-escalation) |
| Question about model performance | `get_model_info` |

## Models

| Model | Feature type | Macro F1 (val) | Speed |
|---|---|---|---|
| TF-IDF + Logistic Regression | Sparse lexical | 0.741 | < 1 ms |
| Fine-tuned FinBERT (`ProsusAI/finbert`) | Domain-adapted transformer | 0.822 | ~1 s |

FinBERT was chosen as the high-accuracy model because it was pre-trained on financial
news and earnings call transcripts, making it domain-aware before fine-tuning even begins.


## 1. Environment Setup

Standard project imports and path resolution. The `src/` module provides reusable
preprocessing helpers (`regex_clean`, `stem_text`) so the agent's fast model applies
exactly the same preprocessing pipeline that was used during training.


In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC_DIR = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FINBERT_MODEL_DIR = PROCESSED_DATA_DIR / 'finbert_final_model'

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from preprocessing import regex_clean, stem_text

print(f'Project root:          {PROJECT_ROOT}')
print(f'FinBERT model dir:     {FINBERT_MODEL_DIR}')
print(f'FinBERT model exists:  {FINBERT_MODEL_DIR.exists()}')


/Users/mehmet/src/text_mining_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root:          /Users/mehmet/src/text_mining_project
FinBERT model dir:     /Users/mehmet/src/text_mining_project/data/processed/finbert_final_model
FinBERT model exists:  True


In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
X = train['text']
y = train['label']

# Recreate the exact same 80/20 stratified split used in tm_tests_10
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=73
)

print(f'Training samples:   {len(X_train)}')
print(f'Validation samples: {len(X_val)}')
print(f'Class distribution  Bearish={int((y_train==0).sum())}  '
      f'Bullish={int((y_train==1).sum())}  Neutral={int((y_train==2).sum())}')


Training samples:   7634
Validation samples: 1909
Class distribution  Bearish=1154  Bullish=1538  Neutral=4942


## 2. Fast Classification Model — TF-IDF + Logistic Regression

The fast model converts each tweet to a sparse TF-IDF vector and classifies it with
a Logistic Regression using class-balanced sample weights. Preprocessing applies the
same `raw_lower_regex_clean_stemmed` pipeline identified as optimal in `tm_tests_10`:
lowercasing, regex removal of URLs and mentions, and Porter stemming.

**Why class-balanced weighting?**
The corpus is heavily skewed toward Neutral (64.7%). Without balancing, the classifier
would over-predict Neutral and miss minority-class signals. The balanced variant
achieved macro F1 0.741 on the held-out validation set, compared to 0.705 for the
default-weighted variant.

This model fits in a few seconds and requires no GPU, making it suitable for
high-throughput or resource-constrained settings.


In [3]:
print('Applying preprocessing to training data...')
X_train_stemmed = X_train.apply(lambda t: stem_text(regex_clean(str(t).lower())))
X_val_stemmed   = X_val.apply(lambda t: stem_text(regex_clean(str(t).lower())))

print('Fitting TF-IDF + balanced Logistic Regression (fast model)...')
fast_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000, class_weight='balanced', random_state=73)),
])
fast_pipeline.fit(X_train_stemmed, y_train)

val_acc = fast_pipeline.score(X_val_stemmed, y_val)
print(f'Fast model ready. Validation accuracy: {val_acc:.4f}')


Applying preprocessing to training data...
Fitting TF-IDF + balanced Logistic Regression (fast model)...
Fast model ready. Validation accuracy: 0.7994


## 3. High-Accuracy Classification Model — Fine-tuned FinBERT

`ProsusAI/finbert` is a BERT-base model pre-trained on financial news corpora. In
`tm_tests_10`, it was fully fine-tuned for three-class sentiment classification
(Bearish/Bullish/Neutral) over 2 epochs with a learning rate of 2×10⁻⁵ on the same
7,634-sample training split. The trained weights were saved to disk and are reloaded
here without retraining.

**Lazy loading pattern**: FinBERT (438 MB) is not loaded at import time. Instead,
`get_finbert()` loads the model on the first call and caches it in module-level
variables `_finbert_model` and `_finbert_tokenizer`. Subsequent calls return the
cached objects immediately, so the one-time startup cost (~10 s) is paid only once
per notebook session.

**Why not always use FinBERT?** For a conversational agent that may receive dozens
of tweets in a session, offering a fast path matters. The TF-IDF model is 1,000× faster
and still reasonably accurate (macro F1 0.741 vs 0.822). The agent uses FinBERT when
precision is important and TF-IDF when speed or exploration is the priority.


In [4]:
LABEL_NAMES = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
DEVICE = (
    'mps'  if torch.backends.mps.is_available() else
    'cuda' if torch.cuda.is_available() else
    'cpu'
)

_finbert_model     = None
_finbert_tokenizer = None


def get_finbert():
    """Lazy-load FinBERT once and cache it for the remainder of the session."""
    global _finbert_model, _finbert_tokenizer
    if _finbert_model is None:
        print('Loading fine-tuned FinBERT from disk (first call only)...')
        _finbert_tokenizer = AutoTokenizer.from_pretrained(str(FINBERT_MODEL_DIR))
        _finbert_model = (
            AutoModelForSequenceClassification
            .from_pretrained(str(FINBERT_MODEL_DIR))
            .to(DEVICE)
        )
        _finbert_model.eval()
        print(f'FinBERT loaded on {DEVICE}.')
    return _finbert_model, _finbert_tokenizer


print(f'Inference device: {DEVICE}')
print('FinBERT will be loaded on the first classify_best() or compare_models() call.')


Inference device: mps
FinBERT will be loaded on the first classify_best() or compare_models() call.


## 4. Agent Tools

The agent has access to four tools. Each tool is a plain Python function that
returns a dictionary — the agent receives this dictionary as a JSON string and
uses the values to formulate its next response.

| Tool | Purpose | Model used |
|---|---|---|
| `classify_fast` | Quick classification | TF-IDF + LogReg |
| `classify_best` | Accurate classification with confidence scores | Fine-tuned FinBERT |
| `compare_models` | Run both models and compare — the core routing tool | Both |
| `get_model_info` | Return validation metrics for both models | — |

### classify_fast
Applies the same preprocessing pipeline used during training (`regex_clean` + `stem_text`),
then scores the preprocessed text through the in-memory TF-IDF + Logistic Regression
pipeline. Returns the predicted label, the sentiment name, and per-class probabilities.
This call completes in under 1 ms.

### classify_best
Tokenises the raw tweet (no stemming — FinBERT operates on natural-language input),
runs a forward pass through the fine-tuned FinBERT model on the available accelerator
(MPS/CUDA/CPU), and applies softmax to the logits to obtain calibrated class probabilities.
Returns the predicted label with per-class confidence scores.

### compare_models
The coordination tool. It calls both `classify_fast` and `classify_best` internally,
checks whether they agree, and returns a structured result that includes both
predictions and a human-readable recommendation. The agent is instructed to call this
tool whenever a tweet is ambiguous **or** when the fast and best models diverge.

### get_model_info
Returns hardcoded validation-set performance metrics for both models. Used when the
user asks which model to trust or wants to understand the accuracy trade-off.


In [5]:
def classify_fast(tweet: str) -> dict:
    """Classify using TF-IDF + Logistic Regression (fast path)."""
    preprocessed = stem_text(regex_clean(str(tweet).lower()))
    label  = int(fast_pipeline.predict([preprocessed])[0])
    probas = fast_pipeline.predict_proba([preprocessed])[0]
    return {
        'label':                label,
        'sentiment':            LABEL_NAMES[label],
        'confidence':           {LABEL_NAMES[i]: round(float(p), 3) for i, p in enumerate(probas)},
        'model':                'TF-IDF + Logistic Regression',
        'macro_f1_validation':  0.741,
    }


def classify_best(tweet: str) -> dict:
    """Classify using fine-tuned FinBERT — most accurate, returns confidence scores."""
    model, tokenizer = get_finbert()
    inputs = tokenizer(
        str(tweet),
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding=True,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
    label = int(logits.argmax(dim=-1).item())
    return {
        'label':                label,
        'sentiment':            LABEL_NAMES[label],
        'confidence':           {LABEL_NAMES[i]: round(float(p), 3) for i, p in enumerate(probs)},
        'model':                'Fine-tuned FinBERT (ProsusAI/finbert)',
        'macro_f1_validation':  0.822,
    }


def compare_models(tweet: str) -> dict:
    """Run both classifiers and return a structured comparison with a recommendation."""
    fast = classify_fast(tweet)
    best = classify_best(tweet)
    agreement = fast['label'] == best['label']
    if agreement:
        recommendation = (
            f"Both models agree: {best['sentiment']}. "
            'Fine-tuned FinBERT result is most reliable.'
        )
    else:
        recommendation = (
            f"Models disagree — fast model: {fast['sentiment']}, "
            f"best model: {best['sentiment']}. "
            'Defer to fine-tuned FinBERT (macro F1 0.822 vs 0.741).'
        )
    return {
        'fast_model':     fast,
        'best_model':     best,
        'agreement':      agreement,
        'recommendation': recommendation,
    }


def get_model_info() -> dict:
    """Return validation-set performance metrics for both available models."""
    return {
        'models': [
            {
                'name':                    'TF-IDF + Logistic Regression',
                'speed':                   'instant (< 1 ms per tweet)',
                'macro_f1_validation':     0.741,
                'weighted_f1_validation':  0.804,
                'accuracy_validation':     0.799,
                'best_use': 'Exploratory analysis, offline use, high-throughput pipelines.',
            },
            {
                'name':                    'Fine-tuned FinBERT (ProsusAI/finbert)',
                'speed':                   '~1 s per tweet (one-time ~10 s load)',
                'macro_f1_validation':     0.822,
                'weighted_f1_validation':  0.865,
                'accuracy_validation':     0.862,
                'per_class_f1':            {'Bearish': 0.758, 'Bullish': 0.799, 'Neutral': 0.910},
                'best_use': 'Production use, high-stakes decisions, imbalanced classes.',
            },
        ],
        'recommendation': (
            'Use fine-tuned FinBERT when accuracy matters. '
            'Use TF-IDF + LogReg when speed or resource constraints apply.'
        ),
    }


print('Tool functions defined: classify_fast, classify_best, compare_models, get_model_info')


Tool functions defined: classify_fast, classify_best, compare_models, get_model_info


## 5. Tool Schemas — OpenAI Function Calling Protocol

Each tool is described to the LLM as a JSON schema following the OpenAI function-calling
specification. The schema tells the model:

- **what the function does** (`description`): a plain-language explanation the model uses
  to decide *when* to call the tool
- **what parameters it expects** (`parameters`): a JSON Schema object the model populates
  with values extracted from the conversation

When the model decides a tool should be called, the API response contains a `tool_calls`
array instead of a plain text reply. The agent loop (Section 6) intercepts these, executes
the corresponding Python function, appends the result as a `tool` role message, and sends
the updated history back to the model — which then formulates the final text response.

This back-and-forth happens entirely within `run_agent()` and is invisible to the user.


In [6]:
TOOL_SCHEMAS = [
    {
        'type': 'function',
        'function': {
            'name': 'classify_fast',
            'description': (
                'Classify a financial tweet using the fast TF-IDF + Logistic Regression model. '
                'Use for quick or exploratory classification. Macro F1 = 0.741 on validation set.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'tweet': {
                        'type': 'string',
                        'description': 'The raw text of the financial tweet to classify.',
                    }
                },
                'required': ['tweet'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'classify_best',
            'description': (
                'Classify a financial tweet using fine-tuned FinBERT, the most accurate model. '
                'Returns the predicted label plus per-class confidence scores. '
                'Macro F1 = 0.822 on validation set.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'tweet': {
                        'type': 'string',
                        'description': 'The raw text of the financial tweet to classify.',
                    }
                },
                'required': ['tweet'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'compare_models',
            'description': (
                'Run both the fast and the best model on the same tweet and compare their '
                'predictions. Use when the tweet seems ambiguous, the user asks for '
                'verification, or you want to check whether the classifiers agree.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {
                    'tweet': {
                        'type': 'string',
                        'description': 'The raw text of the financial tweet to compare.',
                    }
                },
                'required': ['tweet'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_model_info',
            'description': (
                'Return performance metrics and descriptions for both available models. '
                'Use when the user asks about accuracy, reliability, or which model to trust.'
            ),
            'parameters': {
                'type': 'object',
                'properties': {},
                'required': [],
            },
        },
    },
]

print(f'{len(TOOL_SCHEMAS)} tool schemas registered.')


4 tool schemas registered.


## 6. Agent Architecture — Conversation Loop

### System Prompt Design

The system prompt defines the agent's identity, its knowledge of the two models,
and the decision protocol it should follow. Importantly, it does not hardcode which
tool to use for a given tweet — instead, it gives the model a *policy* (when to prefer
fast vs. best vs. compare) and allows the model to reason about which rule applies.
This reasoning step is what makes the workflow agentic rather than a simple lookup.

### Message History

The agent maintains a `history` list of message dictionaries across conversation turns.
Each turn appends the user message, any tool call requests, all tool results, and the
final assistant response. Passing the full history on every API call gives the model
context about prior turns — the agent can refer to a tweet it classified two messages
ago without the user repeating it.

### Tool Execution Loop

The inner `while True` loop inside `run_agent()` implements the standard function-calling
protocol:

```
1. Send history → LLM
2. If response contains tool_calls:
      a. Execute each tool (call the Python function)
      b. Append tool result to history as role='tool'
      c. Go to step 1
3. If response is plain text → return it as the final answer
```

A single user turn may trigger multiple LLM calls if the model chains tool calls
(e.g. first calling `classify_fast`, then deciding to also call `classify_best`).
The loop handles this transparently.

### `_msg_to_dict` Serialisation Helper

The OpenAI SDK returns assistant messages as `ChatCompletionMessage` objects. Before
appending to the history list (which is sent back as plain JSON), these are converted
to serialisable dicts. When the message contains tool calls, the `tool_calls` array
is preserved so the API can match tool results to their originating requests by `id`.


In [7]:
load_dotenv(PROJECT_ROOT / '.env', override=False)

_api_key  = os.getenv('OPENAI_API_KEY') or os.getenv('DEEPSEEK_API_KEY')
_api_base = os.getenv('OPENAI_API_BASE', 'https://api.deepseek.com/v1')
_model    = os.getenv('OPENAI_MODEL', 'deepseek-chat')

if not _api_key:
    raise RuntimeError('No API key found. Set OPENAI_API_KEY or DEEPSEEK_API_KEY in .env')

client = OpenAI(api_key=_api_key, base_url=_api_base)

SYSTEM_PROMPT = (
    'You are a financial tweet sentiment analysis agent. '
    'You classify tweets about financial markets into one of three categories:\n'
    '  - Bearish (0): negative market outlook — decline expectations, negative earnings\n'
    '  - Bullish (1): positive market outlook — growth expectations, positive earnings\n'
    '  - Neutral (2): no clear directional sentiment — factual news, announcements\n\n'
    'You have two classification tools:\n'
    '  - classify_fast:  TF-IDF + Logistic Regression (macro F1 = 0.741, < 1 ms)\n'
    '  - classify_best:  Fine-tuned FinBERT           (macro F1 = 0.822, ~1 s)\n\n'
    'Decision protocol — choose the tool that fits the situation:\n'
    '1. Quick or exploratory classification requests   → classify_fast\n'
    '2. User explicitly asks for most accurate result  → classify_best\n'
    '3. Ambiguous tweet or user asks for verification  → compare_models\n'
    '4. Fast and best classifiers disagree             → compare_models, explain why\n'
    '5. User asks about model performance or accuracy  → get_model_info\n\n'
    'After calling a tool, always explain the result in plain language: '
    'what the sentiment means financially, how confident you are, and any caveats.'
)

TOOL_DISPATCH = {
    'classify_fast':  lambda args: classify_fast(args['tweet']),
    'classify_best':  lambda args: classify_best(args['tweet']),
    'compare_models': lambda args: compare_models(args['tweet']),
    'get_model_info': lambda _:    get_model_info(),
}


def _msg_to_dict(msg) -> dict:
    """Convert a ChatCompletionMessage object to a plain dict for the history list."""
    d = {'role': 'assistant', 'content': msg.content or ''}
    if msg.tool_calls:
        d['tool_calls'] = [
            {
                'id':   tc.id,
                'type': 'function',
                'function': {'name': tc.function.name, 'arguments': tc.function.arguments},
            }
            for tc in msg.tool_calls
        ]
    return d


def run_agent(user_message: str, history: list) -> tuple[str, list]:
    """Send one user turn and return (response_text, updated_history).

    The function loops until the model produces a plain-text response with no
    pending tool calls. Each iteration either executes a batch of tool calls and
    appends their results, or returns the final answer.
    """
    history = history + [{'role': 'user', 'content': user_message}]
    while True:
        response = client.chat.completions.create(
            model=_model,
            messages=[{'role': 'system', 'content': SYSTEM_PROMPT}] + history,
            tools=TOOL_SCHEMAS,
            tool_choice='auto',
        )
        msg = response.choices[0].message
        history = history + [_msg_to_dict(msg)]

        if not msg.tool_calls:
            return msg.content, history

        for tc in msg.tool_calls:
            result = TOOL_DISPATCH[tc.function.name](json.loads(tc.function.arguments))
            history = history + [{
                'role':         'tool',
                'tool_call_id': tc.id,
                'content':      json.dumps(result),
            }]


def chat():
    """Start an interactive conversational session with the agent.

    Maintains full message history across turns so the agent can refer back
    to prior tweets or decisions without the user repeating context.
    """
    history = []
    print('=' * 60)
    print('Financial Sentiment Analysis Agent')
    print("Type a tweet or question. Type 'quit' to exit.")
    print('=' * 60)
    print()
    while True:
        try:
            user_input = input('You: ').strip()
        except (EOFError, KeyboardInterrupt):
            print('\nSession ended.')
            break
        if not user_input:
            continue
        if user_input.lower() in {'quit', 'exit', 'q'}:
            print('Goodbye!')
            break
        response, history = run_agent(user_input, history)
        print(f'\nAgent: {response}\n')


print(f'Agent ready — API: {_api_base}  |  Model: {_model}')


Agent ready — API: https://api.deepseek.com/v1  |  Model: deepseek-chat


## 7. Interactive Session

Uncomment the `chat()` call below to start a live conversation with the agent.
You can type any financial tweet, ask the agent to compare models, or ask questions
about classification accuracy. The session remembers prior turns.

Example prompts to try:
- *'Quickly classify: Tesla stock drops 8% after missing delivery targets.'*
- *'Is this bullish or neutral? Fed holds rates steady as inflation cools toward 2%.'*
- *'Compare both models on: Microsoft beats Q3 earnings but issues cautious Q4 guidance.'*
- *'How accurate are your models?'*


In [8]:
# Uncomment the line below to start an interactive session:
# chat()

print('Uncomment chat() above and run this cell to start an interactive session.')
print('The scripted demo in the next section runs automatically without input.')


Uncomment chat() above and run this cell to start an interactive session.
The scripted demo in the next section runs automatically without input.


## 8. Demo Conversation

The four-turn scripted conversation below exercises all routing paths defined in
the agent's decision protocol:

| Turn | User intent | Expected tool | What it tests |
|---|---|---|---|
| 1 | *'Quickly classify…'* | `classify_fast` | Fast-path routing |
| 2 | *'Double-check with your best model'* | `classify_best` | Accuracy-path routing |
| 3 | *'I'm unsure — bearish or neutral?'* | `compare_models` | Ambiguity routing + disagreement handling |
| 4 | *'Which model should I trust?'* | `get_model_info` | Transparency / model introspection |

Turn 3 is the most revealing: the tweet about the Fed rate decision is genuinely
ambiguous (mixed signals — a potential pause is bullish, persistent inflation is
bearish), and the two models are expected to disagree. The agent should automatically
call `compare_models`, surface the disagreement, and explain why it defers to FinBERT.


In [9]:
DEMO_INPUTS = [
    (
        "Quickly classify this tweet: "
        "'AAPL earnings beat expectations by $0.15 EPS. Stock up 5% in after-hours trading.'"
    ),
    "Can you double-check that with your most accurate model?",
    (
        "I am unsure about this one — could be bearish or neutral. "
        "'Fed signals possible rate pause but warns inflation remains above target. "
        "Analysts are divided on the path forward.'"
    ),
    "Which of your two models should I rely on for important trading decisions?",
]

print('=' * 60)
print('DEMO: Financial Sentiment Analysis Agent')
print('=' * 60)
print()

history = []
for user_input in DEMO_INPUTS:
    print(f'You: {user_input}')
    print()
    response, history = run_agent(user_input, history)
    print(f'Agent: {response}')
    print()
    print('-' * 60)
    print()


DEMO: Financial Sentiment Analysis Agent

You: Quickly classify this tweet: 'AAPL earnings beat expectations by $0.15 EPS. Stock up 5% in after-hours trading.'

Agent: **Result: Bullish (1)** — with **89.5% confidence**.

**Explanation:** The tweet reports that Apple (AAPL) beat earnings expectations by $0.15 per share, and the stock price increased by 5% in after-hours trading. This is a clearly positive financial signal — earnings beats typically indicate strong company performance, and the immediate price jump confirms positive market reaction.

**Model used:** TF-IDF + Logistic Regression (fast mode, macro F1 = 0.741).

**Caveat:** This is the quick model (good for speed, decent accuracy). If you'd like a second opinion with higher accuracy, I can run the more expensive FinBERT model to verify.

------------------------------------------------------------

You: Can you double-check that with your most accurate model?

Loading fine-tuned FinBERT from disk (first call only)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7672.86it/s]


FinBERT loaded on mps.
Agent: Both models are in agreement! Here's the comparison:

| Model | Sentiment | Confidence |
|-------|-----------|------------|
| **Fast** (TF-IDF + LR) | 🟢 Bullish | 89.5% |
| **Best** (FinBERT) | 🟢 Bullish | **99.0%** |

**Final verdict: Bullish (1)** — with very high confidence.

The FinBERT model, which has a stronger macro F1 score of **0.822**, confirms the bullish sentiment with a near-certain **99% confidence**. The tweet's content — an earnings beat ($0.15 EPS above expectations) + a 5% after-hours stock price surge — is as unambiguous a bullish signal as it gets in financial markets.

No caveats needed here; both classifiers strongly agree on the positive outlook. 📈

------------------------------------------------------------

You: I am unsure about this one — could be bearish or neutral. 'Fed signals possible rate pause but warns inflation remains above target. Analysts are divided on the path forward.'

Agent: You were right to be unsure — the two

## 9. Results and Analysis

### Turn-by-Turn Observations

**Turn 1 — `classify_fast` on an unambiguous Bullish tweet**
The AAPL earnings beat is a strong, unambiguous positive signal. The fast model
classified it as Bullish with 89.5% confidence. The agent correctly routed to the
fast tool in response to the word 'quickly', consistent with the decision protocol.

**Turn 2 — `classify_best` for verification**
When asked to double-check, the agent loaded FinBERT (first call, ~10 s startup)
and classified the same tweet as Bullish with 99.0% confidence — a marked increase
over the fast model's 89.5%. Both models agreed, and the agent presented the
comparison as a table showing that the fine-tuned transformer is significantly
more confident on this clear-cut case.

**Turn 3 — `compare_models` on a genuinely ambiguous tweet (model disagreement)**
The Fed rate-pause tweet contains conflicting signals:
- A possible rate pause → positive for risk assets (Bullish)
- Persistent inflation warning → hawkish forward guidance (Bearish)
- Analysts divided → explicit uncertainty signal (Neutral)

The models disagreed: the fast model predicted Bearish with only 41.7% confidence
(barely above its Bullish score of 41.3%), while FinBERT predicted Neutral with
73.4% confidence. The agent automatically called `compare_models`, surfaced the
disagreement, and correctly deferred to FinBERT — which captures the overall
uncertainty of the tweet better than the sparse lexical model.

**Turn 4 — `get_model_info` for decision guidance**
The agent retrieved validation metrics for both models and gave a practical
recommendation: use FinBERT for important decisions, the fast model for bulk
screening. The agent framed the trade-off in trading terms — the cost of a wrong
prediction outweighs the extra second of compute time.

### Key Takeaways

| Observation | Evidence |
|---|---|
| Routing works correctly | Each turn used the expected tool based on user phrasing |
| Disagreement detection works | Turn 3 triggered automatic `compare_models` escalation |
| Context is preserved | Turn 2 correctly referred back to the tweet from Turn 1 |
| FinBERT handles ambiguity better | 73.4% Neutral vs fast model's near-split 41.7% Bearish |

### Limitations

The agent's routing is governed by the LLM's interpretation of the system prompt,
which means edge cases exist: an unusually phrased request might trigger the wrong
tool. A production system would add deterministic pre-processing rules (e.g. always
run `compare_models` if the fast model's top-class probability is below a threshold)
rather than relying entirely on the LLM to decide. Additionally, the decoder-only
LLM (DeepSeek) introduces API latency and cost per turn; a local open-weights model
would eliminate these dependencies.
